# Análise de Variações Algorítmicas e Aprimoramento do VillageNet
**Autor:** Arthur Santos (DCOMP - UFSJ)

Este notebook implementa a arquitetura modular do algoritmo **VillageNet** (Ballal et al., 2026) para avaliar modificações estruturais nas **Etapas 1, 3 e 4**:
- **Experimento 1 (Etapa 4 - Comunidades):** Substituição do WLCF sobre o Grafo Inter-Vilas por *Leiden*, *Spectral Clustering* e *Infomap*.
- **Experimento 2 (Etapas 1 e 3 - Vilas e Fronteira):** Substituição do K-Means simples por *K-Means++*, *MiniBatch K-Means* e *K-Medoids*, além de ponderação contínua por distância nas arestas $A_{UV}$.

## 1. Importação das Bibliotecas

In [13]:
from infomap import Infomap
import igraph as ig
import leidenalg
import numpy as np
import os
import pandas as pd
from scipy.sparse.csgraph import laplacian
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, MiniBatchKMeans, SpectralClustering
from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler
import time

## 2. Funções Auxiliares (K-Medoids e Estimativa Espectral de $k$)
- `_pam_kmedoids`: Implementação leve do K-Medoids (PAM) para testar robustez a *outliers* na Etapa 1.
- `_estimate_k_eigengap`: Estima automaticamente o número de grupos $k$ pelo maior salto nos autovalores (*eigengap*) da matriz Laplaciana normalizada do Grafo Inter-Vilas, mantendo o *Spectral Clustering* 100% não supervisionado.

In [14]:
def _pam_kmedoids(X, n_clusters, max_iter=100, random_state=42):
    """Implementacao leve de K-Medoids (Etapa 1 robusta a outliers) usando L2."""
    rng = np.random.default_rng(random_state)
    N = X.shape[0]
    medoid_indices = rng.choice(N, size=n_clusters, replace=False)
    medoids = X[medoid_indices].copy()

    for _ in range(max_iter):
        dists = cdist(X, medoids, metric='euclidean')
        labels = np.argmin(dists, axis=1)
        new_medoids = medoids.copy()
        for k in range(n_clusters):
            cluster_pts = X[labels == k]
            if len(cluster_pts) > 0:
                intra_dists = cdist(
                    cluster_pts, cluster_pts, metric='euclidean'
                ).sum(axis=1)
                new_medoids[k] = cluster_pts[np.argmin(intra_dists)]
        if np.allclose(medoids, new_medoids):
            break
        medoids = new_medoids

    dists = cdist(X, medoids, metric='euclidean')
    labels = np.argmin(dists, axis=1)
    return labels, medoids


def _estimate_k_eigengap(adjacency_matrix, max_k=25):
    """Estima automaticamente o numero de clusters k via Eigengap da Laplaciana."""
    n_nodes = adjacency_matrix.shape[0]
    max_k = min(max_k, n_nodes - 1)
    if max_k < 2:
        return 1

    L = laplacian(adjacency_matrix, normed=True)
    eigenvalues = np.linalg.eigvalsh(L)
    eigenvalues = np.sort(np.real(eigenvalues))

    # Ignora o primeiro autovalor (~0) e busca o maior salto (eigengap)
    gaps = np.diff(eigenvalues[1 : max_k + 1])
    if len(gaps) == 0:
        return 2
    optimal_k = int(np.argmax(gaps) + 2)
    return optimal_k

## 3. Classe `ModularVillageNet` (As 5 Etapas Plugáveis)
Implementação das equações originais de Ballal et al. (2026) com suporte às variações propostas:
1. **Etapa 1 (`_step1_coarse_graining`):** Cria as $\nu$ vilas (células de Voronoi) via `kmeans_random`, `kmeans++`, `minibatch` ou `kmedoids`.
2. **Etapas 2 e 3 (`_step2_and_3_inter_village_graph`):** Calcula a projeção $\mathcal{D}(\vec{X}_i, U)$ na fronteira entre vilas, seleciona os $\eta$ vizinhos mais próximos via `np.argpartition` e monta a matriz de adjacência $A_{UV}$ (`discrete` ou `continuous`).
3. **Etapa 4 (`_step4_community_detection`):** Detecta comunidades na rede de vilas (`leiden`, `spectral`, `infomap` ou `louvain_igraph`).
4. **Etapa 5 (`fit_predict`):** Funde os pontos das vilas da mesma comunidade.

In [15]:
class ModularVillageNet:

    def __init__(
        self,
        nu=None,
        eta=30,
        village_method='kmeans_random',
        weight_method='discrete',
        community_method='leiden',
        random_state=42,
    ):
        self.nu = nu
        self.eta = eta
        self.village_method = village_method
        self.weight_method = weight_method
        self.community_method = community_method
        self.random_state = random_state

        self.labels_ = None
        self.n_clusters_ = None
        self.step_times_ = {}
        self.total_time_ = 0.0

    def _step1_coarse_graining(self, X, nu):
        if self.village_method == 'kmeans_random':
            km = KMeans(
                n_clusters=nu,
                init='random',
                n_init=1,
                random_state=self.random_state,
            )
            village_labels = km.fit_predict(X)
            centroids = km.cluster_centers_
        elif self.village_method == 'kmeans++':
            km = KMeans(
                n_clusters=nu,
                init='k-means++',
                n_init=10,
                random_state=self.random_state,
            )
            village_labels = km.fit_predict(X)
            centroids = km.cluster_centers_
        elif self.village_method == 'minibatch':
            mb = MiniBatchKMeans(
                n_clusters=nu,
                init='k-means++',
                n_init=3,
                batch_size=1024,
                random_state=self.random_state,
            )
            village_labels = mb.fit_predict(X)
            centroids = mb.cluster_centers_
        elif self.village_method == 'kmedoids':
            village_labels, centroids = _pam_kmedoids(
                X, n_clusters=nu, random_state=self.random_state
            )
        else:
            raise ValueError(
                f'Metodo de vila desconhecido: {self.village_method}'
            )

        return village_labels, centroids

    def _step2_and_3_inter_village_graph(
        self, X, village_labels, centroids, nu, eta
    ):
        A = np.zeros((nu, nu), dtype=np.float64)
        mu_Vi = centroids[village_labels]

        for u in range(nu):
            mask_outside_u = village_labels != u
            idx_outside = np.where(mask_outside_u)[0]
            if len(idx_outside) == 0:
                continue

            X_out = X[idx_outside]
            mu_V_out = mu_Vi[idx_outside]
            mu_U = centroids[u]

            diff_mu = mu_V_out - mu_U
            norm_diff = np.linalg.norm(diff_mu, axis=1, keepdims=True)
            norm_diff = np.where(norm_diff == 0, 1e-12, norm_diff)
            unit_vec = diff_mu / norm_diff

            midpoint = 0.5 * (mu_U + mu_V_out)
            D_Xi_U = np.sum((X_out - midpoint) * unit_vec, axis=1)
            D_Xi_U = np.abs(D_Xi_U)

            k_ext = min(eta, len(D_Xi_U) - 1)
            if k_ext <= 0:
                continue

            ext_local_idx = np.argpartition(D_Xi_U, k_ext)[:k_ext]
            ext_global_idx = idx_outside[ext_local_idx]
            ext_villages = village_labels[ext_global_idx]

            if self.weight_method == 'discrete':
                unique_v, counts = np.unique(ext_villages, return_counts=True)
                for v, c in zip(unique_v, counts):
                    A[u, v] += c
                    A[v, u] += c
            elif self.weight_method == 'continuous':
                ext_dists = D_Xi_U[ext_local_idx]
                sigma = np.median(ext_dists) + 1e-8
                weights = np.exp(-(ext_dists**2) / (2 * (sigma**2)))
                for v, w in zip(ext_villages, weights):
                    A[u, v] += w
                    A[v, u] += w
            else:
                raise ValueError(
                    f'Metodo de peso desconhecido: {self.weight_method}'
                )

        np.fill_diagonal(A, 0.0)
        return A

    def _step4_community_detection(self, A, nu):
        if self.community_method in ['leiden', 'louvain_igraph']:
            sources, targets = np.nonzero(np.triu(A, k=1))
            weights = A[sources, targets].tolist()
            g = ig.Graph(
                n=nu,
                edges=list(zip(sources.tolist(), targets.tolist())),
                directed=False,
            )
            g.es['weight'] = weights

            if self.community_method == 'leiden':
                partition = leidenalg.find_partition(
                    g,
                    leidenalg.ModularityVertexPartition,
                    weights='weight',
                    seed=self.random_state,
                )
            else:
                partition = g.community_multilevel(weights='weight')

            village_communities = np.array(partition.membership, dtype=int)

        elif self.community_method == 'infomap':
            im = Infomap("--two-level --silent")
            sources, targets = np.nonzero(np.triu(A, k=1))
            for u, v in zip(sources, targets):
                im.add_link(int(u), int(v), float(A[u, v]))
            result = im.run()
            village_communities = np.arange(nu, dtype=int)
            for node in im.nodes:
                village_communities[node.node_id] = node.module_id
            _, village_communities = np.unique(
                village_communities, return_inverse=True
            )

        elif self.community_method == 'spectral':
            k_est = _estimate_k_eigengap(A)
            A_reg = A + 1e-6 * np.eye(nu)
            sc = SpectralClustering(
                n_clusters=k_est,
                affinity='precomputed',
                assign_labels='kmeans',
                random_state=self.random_state,
            )
            village_communities = sc.fit_predict(A_reg)
        else:
            raise ValueError(
                f'Metodo de comunidade desconhecido: {self.community_method}'
            )

        return village_communities

    def fit_predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        N = X.shape[0]

        nu = self.nu if self.nu is not None else max(2, min(N // 10, 500))
        eta = min(self.eta, N - 1)

        t0 = time.perf_counter()

        t_s1 = time.perf_counter()
        village_labels, centroids = self._step1_coarse_graining(X, nu)
        self.step_times_['step1_villages'] = time.perf_counter() - t_s1

        t_s23 = time.perf_counter()
        A = self._step2_and_3_inter_village_graph(
            X, village_labels, centroids, nu, eta
        )
        self.step_times_['step2_3_graph'] = time.perf_counter() - t_s23

        t_s4 = time.perf_counter()
        village_communities = self._step4_community_detection(A, nu)
        self.step_times_['step4_community'] = time.perf_counter() - t_s4

        t_s5 = time.perf_counter()
        final_labels = village_communities[village_labels]
        _, final_labels = np.unique(final_labels, return_inverse=True)
        self.step_times_['step5_merging'] = time.perf_counter() - t_s5

        self.total_time_ = time.perf_counter() - t0
        self.labels_ = final_labels
        self.n_clusters_ = len(np.unique(final_labels))

        return self.labels_

## 4. Execução dos Experimentos 1 e 2 nos Datasets de Benchmark
Avaliamos as variações em três bases públicas usadas no artigo original (*Wine*, *Breast Cancer* e *Pen Digits*), medindo:
- **Métricas Externas:** NMI, ARI e erro $|m - m^*|$ na descoberta do número de clusters.
- **Métrica Interna:** Coeficiente de Silhueta.
- **Desempenho:** Tempo de execução em segundos.

In [16]:
datasets = {
    "Wine (m*=3)": (
        StandardScaler().fit_transform(load_wine().data),
        load_wine().target,
        3,
    ),
    "Breast Cancer (m*=2)": (
        StandardScaler().fit_transform(load_breast_cancer().data),
        load_breast_cancer().target,
        2,
    ),
    "Pen Digits (m*=10)": (
        StandardScaler().fit_transform(load_digits().data),
        load_digits().target,
        10,
    ),
}

configurations = [
    # Experimento 1: Variando detector de comunidades (Etapa 4)
    (
        "Exp1: KMeans-Random + Discreto + Louvain",
        "kmeans_random",
        "discrete",
        "louvain_igraph",
    ),
    (
        "Exp1: KMeans-Random + Discreto + Leiden",
        "kmeans_random",
        "discrete",
        "leiden",
    ),
    (
        "Exp1: KMeans-Random + Discreto + Spectral",
        "kmeans_random",
        "discrete",
        "spectral",
    ),
    (
        "Exp1: KMeans-Random + Discreto + Infomap",
        "kmeans_random",
        "discrete",
        "infomap",
    ),
    # Experimento 2: Variando Etapa 1 (Vilas) e Etapa 3 (Pesos)
    ("Exp2: KMeans++ + Discreto + Leiden", "kmeans++", "discrete", "leiden"),
    ("Exp2: MiniBatch + Discreto + Leiden", "minibatch", "discrete", "leiden"),
    ("Exp2: KMedoids + Discreto + Leiden", "kmedoids", "discrete", "leiden"),
    ("Exp2: KMeans++ + Continuo + Leiden", "kmeans++", "continuous", "leiden"),
]

results = []

for ds_name, (X, y_true, m_star) in datasets.items():
    for name, v_meth, w_meth, c_meth in configurations:
        model = ModularVillageNet(
            nu=None,
            eta=30,
            village_method=v_meth,
            weight_method=w_meth,
            community_method=c_meth,
            random_state=42,
        )
        y_pred = model.fit_predict(X)

        nmi = normalized_mutual_info_score(y_true, y_pred)
        ari = adjusted_rand_score(y_true, y_pred)
        sil = (
            silhouette_score(X, y_pred) if model.n_clusters_ > 1 else np.nan
        )
        err_k = abs(model.n_clusters_ - m_star)

        results.append({
            "Dataset": ds_name,
            "Configuracao": name,
            "NMI": round(nmi, 4),
            "ARI": round(ari, 4),
            "Silhueta": round(sil, 4) if not np.isnan(sil) else None,
            "m_inferido": model.n_clusters_,
            "Erro_|m-m*|": err_k,
            "Tempo_s": round(model.total_time_, 4),
        })

df_results = pd.DataFrame(results)
os.makedirs("results", exist_ok=True)
df_results.to_csv("results/benchmark_inicial.csv", index=False)
df_results

/tmp/ipykernel_60914/2755119949.py:147: LegacySurfaceWarning: Infomap.nodes is deprecated and leaves in 3.0; read result.nodes(states=True) off the Result returned by im.run() instead.
  for node in im.nodes:
/tmp/ipykernel_60914/2755119949.py:147: LegacySurfaceWarning: Infomap.nodes is deprecated and leaves in 3.0; read result.nodes(states=True) off the Result returned by im.run() instead.
  for node in im.nodes:
/tmp/ipykernel_60914/2755119949.py:147: LegacySurfaceWarning: Infomap.nodes is deprecated and leaves in 3.0; read result.nodes(states=True) off the Result returned by im.run() instead.
  for node in im.nodes:


,Dataset,Configuracao,NMI,ARI,Silhueta,m_inferido,Erro_|m-m*|,Tempo_s
0,Wine (m*=3),Exp1: KMeans-Random + Discreto + Louvain,0.8920,0.9134,0.2815,3,0,0.0021
1,Wine (m*=3),Exp1: KMeans-Random + Discreto + Leiden,0.8920,0.9134,0.2815,3,0,0.0025
2,Wine (m*=3),Exp1: KMeans-Random + Discreto + Spectral,0.8920,0.9134,0.2815,3,0,0.0051
3,Wine (m*=3),Exp1: KMeans-Random + Discreto + Infomap,0.8920,0.9134,0.2815,3,0,0.0034
4,Wine (m*=3),Exp2: KMeans++ + Discreto + Leiden,0.7438,0.7603,0.2735,3,0,0.0108
5,Wine (m*=3),Exp2: MiniBatch + Discreto + Leiden,0.7402,0.7544,0.2649,3,0,0.0057
6,Wine (m*=3),Exp2: KMedoids + Discreto + Leiden,0.8396,0.8803,0.2793,3,0,0.0025
7,Wine (m*=3),Exp2: KMeans++ + Continuo + Leiden,0.7438,0.7603,0.2735,3,0,0.0112
8,Breast Cancer (m*=2),Exp1: KMeans-Random + Discreto + Louvain,0.4613,0.3970,0.1231,4,2,0.0080
9,Breast Cancer (m*=2),Exp1: KMeans-Random + Discreto + Leiden,0.4156,0.3296,0.1279,4,2,0.0328
